# MOESCORE XACLE 2027 — Full7 end-to-end on Kaggle T4 ×2

**Single notebook:** verify existing Full9 cache → **E5 train/verify** → reuse/extract Whisper large-v2 features on both T4s → **E6 train/verify** → Full7 store → train-only Score Audit → optional Human Score Review → Global Gate train → official validation → label-free test inference → optional post-inference research metrics.

The experimental score-review bypass is explicitly research-only. It skips human review without creating review evidence or claiming human verification. `RUN_SCORE_AUDIT` remains enabled, and the train-only audit still runs. Test labels are read only in the post-inference metrics stage when `RUN_TEST_METRICS=True` and their use is permitted.

**Experimental leakage:** `ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP=True` permits identical train/validation WAV hashes and records affected rows. Validation metrics may be optimistically biased; they are not independent validation and do not establish competition compliance. Test-split duplicate checks remain strict.

Strict review remains the default in the training script. The notebook opts into `EXPERIMENTAL_SKIPPED` only through the central `SKIP_SCORE_REVIEW_EXPERIMENTAL=True` switch and subprocess environment.


## 00 — Central configuration (edit this cell only for environment paths and switches)

The Full9 cache you already have should be attached as a Kaggle **Input dataset**. The default dataset paths come from the notebook you just pushed; correct them if your current Kaggle mount names differ. The repository is pinned to the exact reviewed commit for reproducibility. Keep `WHISPER_BATCH_SIZE=1`.

In [ ]:
from pathlib import Path
import os, sys, json, csv, subprocess, shutil, hashlib, tarfile, time

REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
SOURCE_COMMIT = "fe7de437842e4907421d645cfe71d0015bce895c"
REPO_ROOT = Path("/kaggle/working/MOESCORE")
SKIP_SCORE_REVIEW_EXPERIMENTAL = True
RUN_TEST_METRICS = True
DATA_ROOT = Path("/kaggle/input/datasets/qnv8071/xacle-2027-data/XACLE_Challenge_2027_development_dataset")
FULL9_CACHE_INPUT = Path("/kaggle/input/datasets/qnv8071/moescore-full9-feature-cache")
# Set this to an existing train/validation/test store root if you know the exact path.
FULL9_STORE_OVERRIDE = None

WEIGHTS_DATASET_ROOT = Path("/kaggle/input/datasets/qnv8071/xacle-weights")
WEIGHTS_ROOT = WEIGHTS_DATASET_ROOT / "weights" / "weights"

WORK_ROOT = Path("/kaggle/working/moescore_full7")
CONFIG_ROOT = WORK_ROOT / "configs"
LOG_ROOT = WORK_ROOT / "logs"
WHISPER_ROOT = WORK_ROOT / "whisper_cache"
E5_ROOT = WORK_ROOT / "e5"
E6_ROOT = WORK_ROOT / "e6"
FULL7_ROOT = WORK_ROOT / "full7_store"
GATE_ROOT_BASE = WORK_ROOT / "full7_gate"
GATE_ROOT = GATE_ROOT_BASE
METADATA_ROOT = WORK_ROOT / "whisper_metadata"

# Whisper weights: use a complete local snapshot if already attached, or download once.
WHISPER_SNAPSHOT_INPUT = None  # e.g. '/kaggle/input/.../whisper-large-v2-snapshot'
WHISPER_CACHE_DIR = WORK_ROOT / "hf_whisper" / "hub"
ALLOW_WHISPER_DOWNLOAD = True  # Only if Internet is enabled and the pretrained model is approved.
WHISPER_BATCH_SIZE = 1          # FROZEN: affects decoder padding & mean-pooling contract.
WHISPER_NUM_WORKERS = 0

INSTALL_MISSING_DEPENDENCIES = True
RUN_E5_TRAIN = True
RUN_WHISPER_SMOKE = True
RUN_WHISPER_EXTRACT = True
RUN_E6_TRAIN = True
RUN_FULL7_BUILD = True
RUN_SCORE_AUDIT = True
RUN_GATE_TRAIN = True
RUN_VALIDATION = True
RUN_TEST_INFERENCE = True
ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP = True

# Keep False: this is not a declaration that a human reviewed the scores.
HUMAN_SCORE_REVIEW_CONFIRMED = False
REVIEWER_NAME = ""             # Fill with a real reviewer's name, not a placeholder.
REVIEW_EVIDENCE = ""           # Location/description of the actual reviewed evidence.

# Competition: E6 selects a checkpoint by validation MSE; gate uses validation SRCC.
# Research metrics are provisional until organizer approvals/policy are confirmed.
RUN_PURPOSE = "research_only"  # Not a declaration of challenge submission compliance.
# Experimental overlap does not remove leakage or make validation independent.

E5_DEVICE = "0"
E6_DEVICE = "0"
GATE_DEVICE = "cuda:0"
SHARD_SIZE = 32
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}

# Never upload protected competition data to a public dataset.
CREATE_LOCAL_BACKUP = True
CREATE_PRE_GATE_BACKUP = True
UPLOAD_PRIVATE_PRE_GATE_BACKUP = False
UPLOAD_PRIVATE_BACKUP = False
PRIVATE_BACKUP_HANDLE = "qnv8071/moescore-full7-intermediates"

assert WHISPER_BATCH_SIZE == 1
for p in (WORK_ROOT, CONFIG_ROOT, LOG_ROOT, WHISPER_ROOT, E5_ROOT, E6_ROOT, FULL7_ROOT, GATE_ROOT_BASE, METADATA_ROOT, WHISPER_CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)
print("Full7 work root:", WORK_ROOT)


## 01 — Source + 2×T4 preflight

The Kaggle accelerator must show two CUDA devices. Fail rather than silently running a single GPU. `cuda:0` used by the two Whisper subprocesses is logical after setting `CUDA_VISIBLE_DEVICES` per process.

In [ ]:
if not REPO_ROOT.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", SOURCE_COMMIT], check=True)
current_sha = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
assert current_sha == SOURCE_COMMIT, (current_sha, SOURCE_COMMIT)

required_files = [
    "scripts/build_full7_feature_store.py", "scripts/audit_full7_scores.py",
    "scripts/train_set_a_residual.py", "scripts/train_set_b_residual.py",
    "scripts/extract_whisper_features.py", "scripts/train_moe_gating.py",
    "scripts/evaluate_moe_gating.py", "scripts/infer_moe_gating.py",
    "configs/moe_full7.example.yaml",
]
missing_files = [s for s in required_files if not (REPO_ROOT / s).is_file()]
if missing_files:
    raise FileNotFoundError(f"Expected integrated Full7 source at {SOURCE_COMMIT}: {missing_files}")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select GPU T4 ×2 in Kaggle settings"
print("SOURCE COMMIT", current_sha)
for gpu_idx in range(2):
    props = torch.cuda.get_device_properties(gpu_idx)
    print(f"cuda:{gpu_idx}: {props.name}, {props.total_memory / 2**30:.2f} GiB")
subprocess.run(["nvidia-smi", "-L"], check=True)


## 02 — Kaggle dependencies (reuse the repository's Python-specific installer)

Keep Kaggle's preinstalled torch/torchaudio/torchvision stack. This follows the exact dependency installation strategy used in the latest Full9 notebook. A kernel restart may be necessary if core libraries already imported are replaced. Turn `INSTALL_MISSING_DEPENDENCIES=False` only after a verified installation.

In [ ]:
if INSTALL_MISSING_DEPENDENCIES:
    from scripts.kaggle_full9_dual_gpu import dependency_plan_for_python
    requirements_file, legacy_no_deps = dependency_plan_for_python(sys.version_info)
    installer = [sys.executable, str(REPO_ROOT / "scripts/install_kaggle_dependencies.py"),
                 "--requirements", str(requirements_file),
                 "--constraints", str(WORK_ROOT / "torch_constraints.txt"),
                 "--report-json", str(LOG_ROOT / "dependencies_install.json")]
    subprocess.run(installer, cwd=REPO_ROOT, check=True)
    if legacy_no_deps:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                        "msclap==1.3.4", "laion-clap==1.1.7"], check=True)
print("Dependency setup finished. If packages changed after imports, restart kernel before continuing.")


## 03 — Resolve and fully verify the **existing** Full9 store

This notebook **never** reruns nine expert backbones. The Full9 Kaggle Input may contain either a direct store or the archived `feature_cache.tar` format produced by `scripts/kaggle_full9_artifacts.py`. Archive restore is hash-verified. Source manifests are mandatory because the downstream E5/E6/Core builder needs exact WAV/caption identities.

In [ ]:
from scripts.kaggle_full9_artifacts import restore_feature_cache, validate_full9_store
from data.moe_feature_store import validate_feature_store
from data.set_a_cached_dataset import _read_source_manifest

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Missing official XACLE input: {DATA_ROOT}")
for split in EXPECTED_COUNTS:
    assert (DATA_ROOT / "wav" / split).is_dir(), f"Missing audio directory: {split}"

if FULL9_STORE_OVERRIDE is not None:
    FULL9_ROOT = Path(FULL9_STORE_OVERRIDE).expanduser().resolve()
else:
    if not FULL9_CACHE_INPUT.exists():
        raise FileNotFoundError(f"Attach your existing Full9 cache Dataset: {FULL9_CACHE_INPUT}. Available: {list(Path('/kaggle/input').iterdir())}")
    roots = []
    for candidate_manifest in FULL9_CACHE_INPUT.rglob("train/manifest.json"):
        candidate = candidate_manifest.parent.parent
        if all((candidate / split / "manifest.json").is_file() for split in EXPECTED_COUNTS):
            try:
                body = json.loads(candidate_manifest.read_text())
                if body.get("expert_profile") == "full_9":
                    roots.append(candidate)
            except (OSError, ValueError):
                pass
    if len(roots) == 1:
        FULL9_ROOT = roots[0]
    elif len(roots) > 1:
        raise ValueError(f"Multiple Full9 store roots; set FULL9_STORE_OVERRIDE explicitly: {roots}")
    else:
        # The Kaggle upload format contains feature_cache.tar + upload_manifest.json.
        archives = list(FULL9_CACHE_INPUT.rglob("upload_manifest.json"))
        if len(archives) != 1:
            raise FileNotFoundError("No direct Full9 store; expected exactly one upload_manifest.json + feature_cache.tar. Set FULL9_STORE_OVERRIDE if needed.")
        destination = WORK_ROOT / "full9_restore"
        archive_dir = archives[0].parent
        if not (destination / "feature_cache").is_dir():
            restore_feature_cache(archive_dir, destination)
        roots = []
        for manifest_path in (destination / "feature_cache").rglob("train/manifest.json"):
            candidate = manifest_path.parent.parent
            if all((candidate / split / "manifest.json").is_file() for split in EXPECTED_COUNTS):
                if json.loads(manifest_path.read_text()).get("expert_profile") == "full_9":
                    roots.append(candidate)
        if len(roots) != 1:
            raise ValueError(f"Expected one validated Full9 assembly after restore, found {roots}")
        FULL9_ROOT = roots[0]

full9_report = validate_full9_store(FULL9_ROOT, EXPECTED_COUNTS)
source_records = {}
for split, expected in EXPECTED_COUNTS.items():
    manifest = validate_feature_store(FULL9_ROOT, split=split, require_labels=split != "test")
    records = _read_source_manifest(FULL9_ROOT / split / "source_manifest.json", split=split,
                                    expected_sha256=manifest["source_manifest_sha256"])
    assert len(records) == expected
    if split == "test" and any(record.get("label_raw") is not None for record in records):
        raise ValueError("Full9 test source manifest must be label-free")
    source_records[split] = records

print(json.dumps({"full9_root": str(FULL9_ROOT), "splits": full9_report}, indent=2))
(LOG_ROOT / "full9_verified.json").write_text(json.dumps({"full9_root": str(FULL9_ROOT), "splits": full9_report}, indent=2))


## 04 — Prepare **label-free** Whisper metadata from the signed Full9 source manifest

Do not use the official `test_average.csv` or test scores. The Whisper dataset only needs WAV basename and exact raw caption. Use a stable manifest-derived row order; no inner joins, reordering, or normalization.

In [ ]:
for split, records in source_records.items():
    target = METADATA_ROOT / f"{split}_whisper.csv"
    with target.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["wav_file_name", "text"])
        writer.writeheader()
        for record in records:
            writer.writerow({"wav_file_name": record["wave_file_name"], "text": record["raw_caption"]})
    with target.open(encoding="utf-8", newline="") as handle:
        assert sum(1 for _ in csv.DictReader(handle)) == len(records)
    print(split, "metadata:", target, "rows:", len(records))


## 05 — Train or verify E5 (Set A) from Full9 only

E5 learns the residual head over Top2 M2D + MS-CLAP + MGA features. No backbones and no Whisper. It trains on all official train rows and evaluates, selects, and early-stops on all official validation rows.

In [ ]:
import yaml

def run_logged(name, cmd, *, env=None):
    target = LOG_ROOT / f"{name}.log"
    print("RUN", name, "=>", target, flush=True)
    try:
        with target.open("w", encoding="utf-8") as output:
            subprocess.run([str(v) for v in cmd], cwd=REPO_ROOT, env=env,
                           stdout=output, stderr=subprocess.STDOUT, check=True)
    except subprocess.CalledProcessError:
        print("FAILED", name, "log tail:", target.read_text(errors="replace")[-3000:])
        raise
    print("PASS", name, "tail:", target.read_text(errors="replace")[-1000:], flush=True)

def gpu_env(physical_gpu):
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = str(physical_gpu)
    environment["TOKENIZERS_PARALLELISM"] = "false"
    return environment

E5_CONFIG = CONFIG_ROOT / "e5_full7.yaml"
e5_cfg = yaml.safe_load((REPO_ROOT / "configs/set_a_residual.example.yaml").read_text())
e5_cfg["feature_store_root"] = str(FULL9_ROOT)
e5_cfg["output_dir"] = str(E5_ROOT)
e5_cfg["training"]["selection_metric"] = "official_validation_mse"
e5_cfg["runtime"]["device"] = "cuda"
E5_CONFIG.write_text(yaml.safe_dump(e5_cfg, sort_keys=False))
E5_BEST = E5_ROOT / "expert5_best.pt"
E5_FINAL = E5_ROOT / "expert5_final.pt"
E5_LATEST = E5_ROOT / "expert5_latest.pt"
E5_SELECTION_POLICY = "official_validation_mse_v1"
def e5_checkpoint_policy(path):
    if not path.is_file():
        return None
    payload = torch.load(path, map_location="cpu", weights_only=False)
    return payload.get("training_state", {}).get("selection_policy")

# Old internal-holdout outputs are retrained automatically; they are never passed to resume.
needs_e5_training = RUN_E5_TRAIN and (
    not E5_BEST.is_file() or not E5_FINAL.is_file()
    or e5_checkpoint_policy(E5_BEST) != E5_SELECTION_POLICY
    or e5_checkpoint_policy(E5_FINAL) != E5_SELECTION_POLICY
)
if needs_e5_training:
    command = [sys.executable, REPO_ROOT / "scripts/train_set_a_residual.py", "--config", E5_CONFIG]
    if e5_checkpoint_policy(E5_LATEST) == E5_SELECTION_POLICY:
        command += ["--resume", E5_LATEST]
    run_logged("e5_train", command, env=gpu_env(E5_DEVICE))
if not E5_BEST.is_file() or not E5_FINAL.is_file():
    raise FileNotFoundError("E5 training incomplete: both expert5_best.pt and expert5_final.pt are required before E6/Full7.")
from data.set_a_cached_dataset import SetACachedDataset
from model.expert_adapters.set_a_residual_adapter import SetAResidualAdapter
from model.composite_experts.set_a_residual import SetAResidualHead
from training.set_a_trainer import load_set_a_checkpoint

e5_data = SetACachedDataset(FULL9_ROOT, split="train", require_labels=True)
e5_validation_data = SetACachedDataset(FULL9_ROOT, split="validation", require_labels=True)
e5_model = SetAResidualAdapter(SetAResidualHead(**e5_cfg["model"]))
e5_checkpoint = load_set_a_checkpoint(E5_BEST, model=e5_model, expected_dataset=e5_data)
if RUN_E5_TRAIN and e5_checkpoint["training_state"].get("selection_policy") != E5_SELECTION_POLICY:
    raise ValueError("E5 training did not produce an official-validation checkpoint.")
print("E5 train/validation rows:", len(e5_data), len(e5_validation_data))
print("E5 verified:", E5_BEST, "bytes:", E5_BEST.stat().st_size)
del e5_model, e5_data, e5_validation_data, e5_checkpoint


## 06 — Resolve/download the exact Whisper-large-v2 snapshot once

Both GPU workers read the same immutable snapshot. If Internet is off, attach a complete local Whisper-large-v2 snapshot as a Kaggle Input and set `WHISPER_SNAPSHOT_INPUT`. Never switch the Whisper model variant. The `whisper_top2` runtime records checkpoint file hashes.

In [ ]:
from huggingface_hub import snapshot_download

if WHISPER_SNAPSHOT_INPUT is not None:
    snapshot_path = Path(WHISPER_SNAPSHOT_INPUT).resolve()
    if not (snapshot_path / "config.json").is_file():
        raise FileNotFoundError(f"Whisper local snapshot incomplete: {snapshot_path}")
    whisper_model_ref = str(snapshot_path)
else:
    if ALLOW_WHISPER_DOWNLOAD:
        # Select only PyTorch checkpoint, config and tokenizer assets (not TF/Flax weights).
        snapshot_path = Path(snapshot_download(
            repo_id="openai/whisper-large-v2",
            cache_dir=str(WHISPER_CACHE_DIR), local_files_only=False,
            allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "*.tiktoken"],
        ))
        from model.feature_extractors.whisper_top2 import _local_model_files
        try:
            _local_model_files(snapshot_path)
        except FileNotFoundError:
            # Some HF snapshots publish only PyTorch .bin weights.
            snapshot_path = Path(snapshot_download(
                repo_id="openai/whisper-large-v2",
                cache_dir=str(WHISPER_CACHE_DIR), local_files_only=False,
                allow_patterns=["*.bin", "*.index.json"],
            ))
    else:
        snapshot_path = Path(snapshot_download(
            repo_id="openai/whisper-large-v2", cache_dir=str(WHISPER_CACHE_DIR),
            local_files_only=True,
        ))
    whisper_model_ref = "openai/whisper-large-v2"
assert (snapshot_path / "config.json").is_file()
WHISPER_CONFIG = CONFIG_ROOT / "whisper_full7.yaml"
whisper_cfg = yaml.safe_load((REPO_ROOT / "configs/whisper_top2.example.yaml").read_text())
whisper_cfg["whisper_top2"]["model_name_or_path"] = whisper_model_ref
whisper_cfg["runtime"].update({"device": "auto", "offline": True, "cache_dir": str(WHISPER_CACHE_DIR)})
WHISPER_CONFIG.write_text(yaml.safe_dump(whisper_cfg, sort_keys=False))
print("Whisper snapshot:", snapshot_path)
print("Whisper config:", WHISPER_CONFIG)


## 07 — Whisper smoke on both physical T4 GPUs

Each worker has `CUDA_VISIBLE_DEVICES` set to one physical GPU, and internally uses logical `cuda:0`; this avoids real E6 cache metadata differing between train and validation. Always use batch size **1**, including resumed runs.

In [ ]:
def whisper_command(split, *, smoke=False):
    cmd = [sys.executable, str(REPO_ROOT / "scripts/extract_whisper_features.py"),
           "--config", str(WHISPER_CONFIG),
           "--metadata-csv", str(METADATA_ROOT / f"{split}_whisper.csv"),
           "--audio-dir", str(DATA_ROOT / "wav" / split),
           "--split", split,
           "--output-dir", str(WHISPER_ROOT),
           "--batch-size", str(WHISPER_BATCH_SIZE),
           "--num-workers", str(WHISPER_NUM_WORKERS)]
    if smoke:
        cmd += ["--smoke-test"]
    return cmd

if RUN_WHISPER_SMOKE:
    run_logged("whisper_train_gpu0_smoke", whisper_command("train", smoke=True), env=gpu_env(0))
    run_logged("whisper_validation_gpu1_smoke", whisper_command("validation", smoke=True), env=gpu_env(1))
print("Whisper smoke complete. Verify [1,1280] audio and text shapes in both logs.")


## 08 — Dual-T4 Whisper extraction with safe split isolation and resumable caches

GPU 0 extracts **train**. GPU 1 extracts **validation then test**, sequentially, while train runs independently. Two *subprocesses* keep GPU contexts isolated. Each split writes only to `whisper_cache/<split>` and supports resuming a validated prefix. Log files can be inspected if Kaggle stops a worker.

In [ ]:
def start_whisper(split, physical_gpu):
    log_path = LOG_ROOT / f"whisper_{split}_gpu{physical_gpu}.log"
    stream = log_path.open("a", encoding="utf-8")
    stream.write(f"\n== START {split} on physical GPU {physical_gpu}: {time.ctime()} ==\n")
    stream.flush()
    process = subprocess.Popen([str(v) for v in whisper_command(split)],
                               cwd=REPO_ROOT, env=gpu_env(physical_gpu),
                               stdout=stream, stderr=subprocess.STDOUT)
    return process, stream, log_path

if RUN_WHISPER_EXTRACT:
    train_process, train_log_stream, train_log_path = start_whisper("train", 0)
    val_process, val_log_stream, val_log_path = start_whisper("validation", 1)
    try:
        val_rc = val_process.wait()
        val_log_stream.close()
        test_rc = None
        if val_rc == 0:
            test_process, test_log_stream, test_log_path = start_whisper("test", 1)
            try:
                test_rc = test_process.wait()
            finally:
                test_log_stream.close()
        train_rc = train_process.wait()
    finally:
        if train_process.poll() is None:
            train_process.terminate()
            train_process.wait()
        train_log_stream.close()
        if val_process.poll() is None:
            val_process.terminate()
            val_process.wait()
        val_log_stream.close()
    statuses = {"train": train_rc, "validation": val_rc, "test": test_rc}
    print("Whisper exit codes:", statuses)
    if statuses != {"train": 0, "validation": 0, "test": 0}:
        for name in ("train", "validation", "test"):
            paths = sorted(LOG_ROOT.glob(f"whisper_{name}_gpu*.log"))
            if paths:
                print(f"{name} log tail:\n{paths[-1].read_text(errors='replace')[-2000:]}")
        raise RuntimeError(f"Whisper extraction failed; validated prefixes remain for resume: {statuses}")
else:
    print("Whisper extraction skipped by configuration; existing caches still require verification.")


## 09 — Validate all Whisper splits and Full9 ↔ Whisper alignment

This step reads the actual original WAV bytes and tests their hashes through `SetBCachedDataset`, verifies order and 1280+1280 dimensions, and catches GPU/preprocessing provenance divergence before E6 training. Do not patch the cache if validation fails.

In [ ]:
from data.set_b_cached_dataset import SetBCachedDataset
whisper_datasets = {}
for split, expected in EXPECTED_COUNTS.items():
    dataset = SetBCachedDataset(FULL9_ROOT, WHISPER_ROOT, split=split,
                                source_audio_dir=DATA_ROOT / "wav" / split)
    assert len(dataset) == expected
    sample = dataset[0]
    assert sample["whisper_audio"].shape == (1280,)
    assert sample["whisper_text"].shape == (1280,)
    whisper_datasets[split] = dataset
    print(f"Whisper {split} PASS: {len(dataset)} rows, sample={sample['sample_id']}")
train_dataset = whisper_datasets["train"]
for split in ("validation", "test"):
    current = whisper_datasets[split]
    assert train_dataset.whisper_checkpoint_identity == current.whisper_checkpoint_identity, "Whisper checkpoint differs"
    assert train_dataset.whisper_preprocessing_identity == current.whisper_preprocessing_identity, "Whisper preprocessing differs"
print("Whisper E6 provenance and alignment PASS")
del whisper_datasets


## 10 — Train or verify E6 (Set B) using Full9 + Whisper

E6 has a fixed 30-epoch training policy; its existing code selects the best checkpoint by validation MSE. This behavior is **provisional for challenge compliance** and must be clarified with organizers. Use the existing checkpoint loader, not a hand-built substitute.

In [ ]:
E6_CONFIG = CONFIG_ROOT / "e6_full7.yaml"
e6_cfg = yaml.safe_load((REPO_ROOT / "configs/set_b_residual.example.yaml").read_text())
e6_cfg["feature_store_root"] = str(FULL9_ROOT)
e6_cfg["whisper_cache_root"] = str(WHISPER_ROOT)
e6_cfg["output_dir"] = str(E6_ROOT)
e6_cfg["runtime"]["device"] = "cuda"
e6_cfg["training"]["allow_train_validation_audio_overlap"] = ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
e6_cfg["source_audio_dirs"] = {s: str(DATA_ROOT / "wav" / s) for s in EXPECTED_COUNTS}
E6_CONFIG.write_text(yaml.safe_dump(e6_cfg, sort_keys=False))
E6_BEST = E6_ROOT / "expert6_best.pt"
E6_LATEST = E6_ROOT / "expert6_latest.pt"
E6_FINAL = E6_ROOT / "expert6_final.pt"
from data.set_b_cached_dataset import SetBCachedDataset
e6_expected_datasets = {
    split: SetBCachedDataset(FULL9_ROOT, WHISPER_ROOT, split=split)
    for split in ("train", "validation")
}
e6_expected_provenance = {split: dataset.provenance for split, dataset in e6_expected_datasets.items()}
from data.audio_overlap import build_train_validation_audio_overlap_diagnostic
e6_expected_overlap_diagnostic = build_train_validation_audio_overlap_diagnostic(
    e6_expected_datasets["train"].records, e6_expected_datasets["validation"].records,
    allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
)
def e6_checkpoint_matches_experiment(path):
    if not path.is_file():
        return False
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except Exception:
        return False
    settings = payload.get("training_settings")
    return (payload.get("format_version") == 2
            and isinstance(settings, dict)
            and settings.get("allow_train_validation_audio_overlap") == ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
            and settings.get("selection_policy") == "official_validation_mse_min_v1"
            and payload.get("dataset_provenance") == e6_expected_provenance
            and payload.get("audio_overlap_diagnostic") == e6_expected_overlap_diagnostic
            and payload.get("audio_overlap_diagnostic_sha256") == e6_expected_overlap_diagnostic["diagnostic_sha256"])
e6_complete_for_experiment = all(e6_checkpoint_matches_experiment(path) for path in (E6_BEST, E6_FINAL))
e6_latest_can_resume = e6_checkpoint_matches_experiment(E6_LATEST)
if e6_latest_can_resume:
    e6_latest_can_resume = torch.load(E6_LATEST, map_location="cpu", weights_only=False).get("epoch", 30) < 30
# Legacy or strict-policy checkpoints are kept for inspection but do not satisfy this experiment.
if RUN_E6_TRAIN and not e6_complete_for_experiment:
    if RUN_PURPOSE != "research_only":
        print("WARNING: Confirm organizer validation-driven checkpoint selection approval before challenge submission.")
    command = [sys.executable, REPO_ROOT / "scripts/train_set_b_residual.py", "--config", E6_CONFIG]
    if e6_latest_can_resume:
        command += ["--resume", E6_LATEST]
    run_logged("e6_train", command, env=gpu_env(E6_DEVICE))
if not E6_BEST.is_file() or not E6_FINAL.is_file():
    raise FileNotFoundError("E6 training incomplete: require both expert6_best.pt and expert6_final.pt (30 epochs).")
from model.composite_experts.set_b_residual import SetBResidualHead
from model.expert_adapters.set_b_residual_adapter import SetBResidualAdapter
from training.set_b_trainer import load_set_b_checkpoint
verified_e6_model = SetBResidualAdapter(SetBResidualHead(**e6_cfg["model"]))
load_set_b_checkpoint(E6_BEST, model=verified_e6_model,
                      expected_dataset=SetBCachedDataset(FULL9_ROOT, WHISPER_ROOT, split="train"))
print("E6 verified:", E6_BEST, "bytes:", E6_BEST.stat().st_size)
del verified_e6_model


## 11 — Assemble and validate verified Full7 v3 (CPU; no backbones)

Uses *exact* existing CLI: Full9 → E5/E6 frozen heads → seven score columns and **29,959 features** in E1…E7 order. It binds source WAV/Whisper/E5/E6 provenance and rejects inconsistent labels or cache fingerprints.

In [ ]:
from data.moe_feature_store import validate_feature_store
FULL7_BUILD_REPORT = LOG_ROOT / "full7_build.json"
if RUN_FULL7_BUILD:
    run_logged("full7_build", [
        sys.executable, REPO_ROOT / "scripts/build_full7_feature_store.py",
        "--full9-root", FULL9_ROOT,
        "--whisper-root", WHISPER_ROOT,
        "--e5-checkpoint", E5_BEST,
        "--e6-checkpoint", E6_BEST,
        "--source-audio-root", DATA_ROOT / "wav",
        "--output-root", FULL7_ROOT,
        "--split", "train", "--split", "validation", "--split", "test",
        "--shard-size", str(SHARD_SIZE),
        "--report-json", FULL7_BUILD_REPORT,
    ] + (["--allow-train-validation-audio-overlap"]
         if ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP else []))
for split, count in EXPECTED_COUNTS.items():
    manifest = validate_feature_store(FULL7_ROOT, split=split, require_labels=split != "test")
    actual = sum(shard["count"] for shard in manifest["shards"])
    assert (manifest["expert_profile"], manifest["feature_width"], manifest["score_width"], actual) == ("full_7", 29959, 7, count)
    assert manifest["labels_present"] is (split != "test")
    print(split, "FULL7 PASS", actual, manifest["store_identity_sha256"])


## 12 — Train-only score audit and **feasibility** diagnostics

Score range alone does not imply MOS calibration: E1/E2 are similarity proxies, E3 regression scale must be verified using real checkpoint evidence, and E5/E6 residual scores must retain their learned scale. This cell writes review hashes; it **does not approve** any scale semantics.

In [ ]:
SCORE_AUDIT_PATH = LOG_ROOT / "full7_score_audit.json"
if RUN_SCORE_AUDIT:
    run_logged("full7_audit", [sys.executable, REPO_ROOT / "scripts/audit_full7_scores.py",
                                "--store-root", FULL7_ROOT, "--output-json", SCORE_AUDIT_PATH])
if not SCORE_AUDIT_PATH.is_file():
    raise FileNotFoundError("Score audit missing; run train-only audit before training.")
audit = json.loads(SCORE_AUDIT_PATH.read_text())
assert audit["status"] == "PASS" and audit["split"] == "train"
print("AUDIT HASHES", {k: audit[k] for k in ("store_identity_sha256", "score_contract_sha256", "score_audit_sha256")})
for name, stats in audit["diagnostics"]["experts"].items():
    print(f"{name}: min={stats['min']:.4f}, max={stats['max']:.4f}, mean={stats['mean']:.4f}, std={stats['std']:.4f}, train Pearson={stats['train_label_diagnostics']['pearson']}")

# A convex weighted sum cannot predict outside the per-row [min(experts), max(experts)] interval.
import numpy as np
from data.moe_feature_store import MoeFeatureStoreDataset
train_store = MoeFeatureStoreDataset(FULL7_ROOT, split="train", require_labels=True)
score_values = np.stack([train_store[i]["scores"].numpy() for i in range(len(train_store))])
raw_labels = np.asarray([float(train_store[i]["label_raw"]) for i in range(len(train_store))], dtype=np.float32)
targets_norm = (raw_labels - 5.0) / 5.0
unreachable = (targets_norm < score_values.min(axis=1)) | (targets_norm > score_values.max(axis=1))
feasibility = {"train_rows": int(len(unreachable)), "fraction_outside_expert_score_envelope": float(unreachable.mean())}
(LOG_ROOT / "score_interval_feasibility.json").write_text(json.dumps(feasibility, indent=2))
print("Gate score-interval feasibility:", feasibility)


## 12A — Preserve E5/E6 + Whisper before the review-policy stage

The GPU-expensive stages are already complete. Package **Whisper + E5 + E6** (not the large Full7 cache, which can be rebuilt from them) so the first Run All can preserve expensive work even if the later score-review gate intentionally stops training. Optional Kaggle upload is **private-only** and off by default.

In [ ]:
if CREATE_PRE_GATE_BACKUP:
    from scripts.kaggle_full9_artifacts import check_private_upload_target
    PRE_GATE_BACKUP = WORK_ROOT / "pre_gate_backup"
    PRE_GATE_BACKUP.mkdir(parents=True, exist_ok=True)
    archive = PRE_GATE_BACKUP / "whisper_e5_e6.tar"
    temporary_archive = PRE_GATE_BACKUP / "whisper_e5_e6.tar.tmp"
    with tarfile.open(temporary_archive, "w") as tar:
        for resource in (WHISPER_ROOT, E5_ROOT, E6_ROOT):
            for file in resource.rglob("*"):
                if file.is_file():
                    tar.add(file, arcname=str(file.relative_to(WORK_ROOT)), recursive=False)
    temporary_archive.replace(archive)
    sha = hashlib.sha256()
    with archive.open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""):
            sha.update(block)
    (PRE_GATE_BACKUP / "manifest.json").write_text(json.dumps({
        "source_commit": SOURCE_COMMIT,
        "full9_root_reference": str(FULL9_ROOT),
        "whisper_e5_e6_sha256": sha.hexdigest(),
        "archive_bytes": archive.stat().st_size,
    }, indent=2))
    print("PRE-GATE BACKUP:", archive, "bytes:", archive.stat().st_size)
    if UPLOAD_PRIVATE_PRE_GATE_BACKUP:
        import kagglehub
        handle = "qnv8071/moescore-full7-pre-gate-artifacts"
        check_private_upload_target(handle)
        kagglehub.dataset_upload(handle=handle,
                                 local_dataset_dir=str(PRE_GATE_BACKUP),
                                 version_notes="Whisper large-v2 and E5/E6 verified artifacts")
        print("Private backup uploaded:", handle)


## 13 — Human Score Review (strict mode only)

The train-only Score Audit in stage 12 always runs. With `SKIP_SCORE_REVIEW_EXPERIMENTAL=True`, this stage logs that human review was skipped and continues without writing a review file or reviewer/evidence fields. With the switch False, the existing genuine reviewer and evidence requirements remain in force; the review file is bound to the current audit hashes.


In [ ]:
SCORE_REVIEW_PATH = CONFIG_ROOT / "score_review.json"
if SKIP_SCORE_REVIEW_EXPERIMENTAL:
    print("\n*** EXPERIMENTAL: HUMAN SCORE REVIEW SKIPPED ***")
    print("The train-only Score Audit ran at:", SCORE_AUDIT_PATH)
    print("No reviewer, evidence, or score_review.json is being fabricated.")
    print("Gate results will be marked EXPERIMENTAL_SKIPPED / research_only.\n")
else:
    review_body = {
        "schema_version": 1,
        "train_store_identity_sha256": audit["store_identity_sha256"],
        "score_contract_sha256": audit["score_contract_sha256"],
        "score_audit_sha256": audit["score_audit_sha256"],
        "reviewer": REVIEWER_NAME,
        "evidence": REVIEW_EVIDENCE,
        "experts": {f"expert{i}": "UNREVIEWED" for i in range(1, 8)},
    }
    (CONFIG_ROOT / "score_review_template.json").write_text(json.dumps(review_body, indent=2))
    if HUMAN_SCORE_REVIEW_CONFIRMED:
        if not REVIEWER_NAME.strip() or not REVIEW_EVIDENCE.strip():
            raise ValueError("Provide real REVIEWER_NAME and REVIEW_EVIDENCE before confirming review")
        review_body["experts"] = {f"expert{i}": "REVIEWED_FOR_GATE_PROXY" for i in range(1, 8)}
        SCORE_REVIEW_PATH.write_text(json.dumps(review_body, indent=2))
        print("Verified human-review declaration written:", SCORE_REVIEW_PATH)
    else:
        print("REVIEW REQUIRED: inspect audit and sources; provide genuine reviewer/evidence, then rerun this cell.")
        print("Template:", CONFIG_ROOT / "score_review_template.json")


## 14 — Full7 Gate configuration and training on a single T4

The gate is `29959 → 512 → 7`; optimizer updates **only the gate**. Reuses Core's AdamW + ClippedMSE/Contrastive/Diversity loss and strict checkpoint v3. The notebook refuses real training without signed score-review evidence. Best selection by validation SRCC remains **provisional** for competition approval.

In [ ]:
GATE_CONFIG = CONFIG_ROOT / "moe_full7.yaml"
SCORE_REVIEW_POLICY = "EXPERIMENTAL_SKIPPED" if SKIP_SCORE_REVIEW_EXPERIMENTAL else "HUMAN_REVIEWED"
if SKIP_SCORE_REVIEW_EXPERIMENTAL:
    print("Gate review policy:", SCORE_REVIEW_POLICY, "(research_only; no human review claimed)")
elif not HUMAN_SCORE_REVIEW_CONFIRMED or not SCORE_REVIEW_PATH.is_file():
    raise RuntimeError("STOP: strict mode requires genuine Human Score Review evidence bound to the train-only audit.")

from data.audio_overlap import build_train_validation_audio_overlap_diagnostic
from data.moe_feature_store import validate_feature_store
from data.moe_training_dataset import MoeTrainingDataset
from training.config import load_moe_training_config
from training.checkpoint_manager import full7_source_code_sha256
from scripts.train_moe_gating import _checkpoint_config, _store_provenance

train_store_manifest = validate_feature_store(FULL7_ROOT, split="train", require_labels=True)
validation_store_manifest = validate_feature_store(FULL7_ROOT, split="validation", require_labels=True)
current_overlap_diagnostic = build_train_validation_audio_overlap_diagnostic(
    train_store_manifest["assembly_identity"]["ordered_source_fingerprints"],
    validation_store_manifest["assembly_identity"]["ordered_source_fingerprints"],
    allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
)
train_gate_dataset = MoeTrainingDataset(FULL7_ROOT, split="train", require_labels=True)
validation_gate_dataset = MoeTrainingDataset(FULL7_ROOT, split="validation", require_labels=True)
expected_expert_provenance, expected_feature_store_identity = _store_provenance(
    train_gate_dataset, validation_gate_dataset,
    allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
)
expected_feature_store_identity["score_review_policy"] = SCORE_REVIEW_POLICY
expected_score_review_sha256 = None
if SCORE_REVIEW_POLICY == "HUMAN_REVIEWED":
    expected_score_review_sha256 = hashlib.sha256(SCORE_REVIEW_PATH.read_bytes()).hexdigest()
    expected_feature_store_identity["score_review_sha256"] = expected_score_review_sha256

# Output paths are removed from the checkpoint configuration fingerprint, so the run key
# binds policy, model configuration, source stores, and overlap diagnostics deterministically.
gate_cfg = yaml.safe_load((REPO_ROOT / "configs/moe_full7.example.yaml").read_text())
gate_cfg["runtime"]["device"] = GATE_DEVICE
gate_cfg["data"]["feature_store_root"] = str(FULL7_ROOT)
gate_cfg["data"]["allow_train_validation_audio_overlap"] = ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
gate_cfg["data"]["score_review_file"] = str(SCORE_REVIEW_PATH)
gate_cfg["output_dir"] = str(GATE_ROOT_BASE / "candidate")
GATE_CONFIG.write_text(yaml.safe_dump(gate_cfg, sort_keys=False))
raw_gate_config = load_moe_training_config(GATE_CONFIG)
raw_gate_config["data"]["score_review_policy"] = SCORE_REVIEW_POLICY
expected_gate_config = _checkpoint_config(raw_gate_config)

def canonical_sha256(value):
    return hashlib.sha256(json.dumps(
        value, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False,
    ).encode()).hexdigest()

expected_gate_config_sha256 = canonical_sha256(expected_gate_config)
run_identity = {
    "score_review_policy": SCORE_REVIEW_POLICY,
    "score_review_sha256": expected_score_review_sha256,
    "gate_config_sha256": expected_gate_config_sha256,
    "train_store_identity_sha256": train_store_manifest["store_identity_sha256"],
    "validation_store_identity_sha256": validation_store_manifest["store_identity_sha256"],
    "audio_overlap_diagnostic": current_overlap_diagnostic,
}
run_key = canonical_sha256(run_identity)[:16]
GATE_ROOT = GATE_ROOT_BASE / f"{SCORE_REVIEW_POLICY.lower()}_{run_key}"

def write_gate_config(output_root):
    gate_cfg["output_dir"] = str(output_root)
    GATE_CONFIG.write_text(yaml.safe_dump(gate_cfg, sort_keys=False))

write_gate_config(GATE_ROOT)
current_source_sha256 = full7_source_code_sha256()
latest = GATE_ROOT / "latest_gate.pt"
best = GATE_ROOT / "best_gate.pt"
completed = GATE_ROOT / "run_manifest.json"
completed_manifest = json.loads(completed.read_text()) if completed.is_file() else {}

def gate_checkpoint_compatible(payload):
    identity = payload.get("feature_store_identity", {})
    return (
        payload.get("format_version") == 4
        and payload.get("selection_policy") == "official_validation_srcc_strict_v1_provisional"
        and payload.get("config") == expected_gate_config
        and payload.get("config_sha256") == expected_gate_config_sha256
        and payload.get("config_sha256") == canonical_sha256(payload.get("config"))
        and identity == expected_feature_store_identity
        and payload.get("feature_store_identity_sha256") == canonical_sha256(identity)
        and payload.get("expert_checkpoint_provenance") == expected_expert_provenance
        and payload.get("expert_checkpoint_provenance_sha256") == canonical_sha256(expected_expert_provenance)
        and payload.get("source_code_sha256") == current_source_sha256
        and identity.get("audio_overlap_diagnostic") == current_overlap_diagnostic
        and identity.get("score_review_policy") == SCORE_REVIEW_POLICY
    )

best_compatible = False
if best.is_file():
    try:
        best_compatible = gate_checkpoint_compatible(torch.load(best, map_location="cpu", weights_only=True))
    except Exception:
        best_compatible = False
latest_compatible = False
if latest.is_file():
    try:
        latest_compatible = gate_checkpoint_compatible(torch.load(latest, map_location="cpu", weights_only=True))
    except Exception:
        latest_compatible = False

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None

completed_compatible = (
    best_compatible
    and completed_manifest.get("source_code_sha256") == current_source_sha256
    and completed_manifest.get("selection_policy") == "official_validation_srcc_strict_v1_provisional"
    and completed_manifest.get("score_review_policy") == SCORE_REVIEW_POLICY
    and completed_manifest.get("human_review_performed") == (SCORE_REVIEW_POLICY == "HUMAN_REVIEWED")
    and completed_manifest.get("run_purpose") == RUN_PURPOSE
    and completed_manifest.get("score_review_sha256") == expected_score_review_sha256
    and completed_manifest.get("allow_train_validation_audio_overlap") == ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
    and completed_manifest.get("audio_overlap_diagnostic") == current_overlap_diagnostic
    and completed_manifest.get("config_sha256") == expected_gate_config_sha256
    and completed_manifest.get("train_store_identity_sha256") == train_store_manifest["store_identity_sha256"]
    and completed_manifest.get("validation_store_identity_sha256") == validation_store_manifest["store_identity_sha256"]
    and completed_manifest.get("gate_checkpoint_sha256") == file_sha256(best)
)
resume_checkpoint = latest if latest_compatible else None
has_existing_checkpoint = latest.is_file() or best.is_file()
if has_existing_checkpoint and not completed_compatible and not (latest_compatible and best_compatible):
    previous_root = GATE_ROOT
    GATE_ROOT = GATE_ROOT_BASE / f"{previous_root.name}_fresh_{time.time_ns()}"
    write_gate_config(GATE_ROOT)
    latest = GATE_ROOT / "latest_gate.pt"
    best = GATE_ROOT / "best_gate.pt"
    completed = GATE_ROOT / "run_manifest.json"
    completed_compatible = False
    print("Incompatible Gate artifacts preserved at", previous_root,
          "; using isolated output directory", GATE_ROOT)

if RUN_GATE_TRAIN:
    if completed_compatible:
        print("Compatible Gate run manifest and best checkpoint already exist: evaluate/infer without retraining.")
    else:
        cmd = [sys.executable, REPO_ROOT / "scripts/train_moe_gating.py",
               "--config", GATE_CONFIG, "--device", GATE_DEVICE,
               "--report-json", LOG_ROOT / f"full7_training_{GATE_ROOT.name}.json"]
        if resume_checkpoint is not None:
            cmd += ["--resume", resume_checkpoint]
        gate_env = gpu_env(0)
        if SKIP_SCORE_REVIEW_EXPERIMENTAL:
            gate_env["MOESCORE_EXPERIMENTAL_SKIP_SCORE_REVIEW"] = "1"
        else:
            gate_env.pop("MOESCORE_EXPERIMENTAL_SKIP_SCORE_REVIEW", None)
        run_logged("full7_gate_train", cmd, env=gate_env)
else:
    print("Gate training skipped (RUN_GATE_TRAIN=False).")


## 15 — Official validation evaluation (no validation gradients)

In [ ]:
GATE_BEST = GATE_ROOT / "best_gate.pt"
if RUN_VALIDATION:
    if not GATE_BEST.is_file():
        raise FileNotFoundError("Full7 best_gate.pt missing; complete training first")
    run_logged("full7_validation", [sys.executable, REPO_ROOT / "scripts/evaluate_moe_gating.py",
                "--config", GATE_CONFIG, "--checkpoint", GATE_BEST,
                "--device", GATE_DEVICE, "--report-json", LOG_ROOT / "validation_metrics.json"],
               env=gpu_env(0))
    print("Validation report:", (LOG_ROOT / "validation_metrics.json").read_text()[:3000])


## 16 — Label-free test inference, then optional research metrics

Inference writes and freezes `wave_file_name,pred_score` without reading test labels. The immediately following cell may read `meta_data/test_average.csv` only after inference has completed, and only when `RUN_TEST_METRICS=True` and test-label use is permitted. Metrics are marked `POST_INFERENCE_RESEARCH_ONLY`; they do not affect training, selection, validation, or inference.

In [ ]:
SUBMISSION = GATE_ROOT / "submission.csv"
TEST_INFERENCE_COMPLETED = False
TEST_PREDICTION_SNAPSHOT = None
if RUN_TEST_INFERENCE:
    if not GATE_BEST.is_file():
        raise FileNotFoundError("Full7 best gate checkpoint missing")
    run_logged("full7_test_inference", [sys.executable, REPO_ROOT / "scripts/infer_moe_gating.py",
                "--config", GATE_CONFIG, "--checkpoint", GATE_BEST,
                "--device", GATE_DEVICE, "--output", SUBMISSION,
                "--report-json", LOG_ROOT / "test_inference.json"], env=gpu_env(0))
    with SUBMISSION.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        assert reader.fieldnames == ["wave_file_name", "pred_score"]
        rows = list(reader)
    assert len(rows) == EXPECTED_COUNTS["test"]
    TEST_PREDICTION_SNAPSHOT = LOG_ROOT / f"test_predictions_frozen_{time.time_ns()}.csv"
    shutil.copy2(SUBMISSION, TEST_PREDICTION_SNAPSHOT)
    TEST_PREDICTION_SNAPSHOT.chmod(0o444)
    submission_sha256 = hashlib.sha256(SUBMISSION.read_bytes()).hexdigest()
    frozen_sha256 = hashlib.sha256(TEST_PREDICTION_SNAPSHOT.read_bytes()).hexdigest()
    assert submission_sha256 == frozen_sha256
    TEST_INFERENCE_COMPLETED = True
    print("Submission rows:", len(rows), "file:", SUBMISSION)
    print("Frozen prediction snapshot:", TEST_PREDICTION_SNAPSHOT, frozen_sha256)
    print("First two predictions:", rows[:2])


In [ ]:
TEST_METRICS_PATH = LOG_ROOT / "test_metrics.json"
if not RUN_TEST_METRICS:
    test_metrics_report = {
        "status": "SKIPPED",
        "run_purpose": "research_only",
        "reason": "RUN_TEST_METRICS=False; test labels were not accessed.",
    }
    TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report, indent=2) + "\n")
elif not TEST_INFERENCE_COMPLETED or TEST_PREDICTION_SNAPSHOT is None:
    test_metrics_report = {
        "status": "SKIPPED",
        "run_purpose": "research_only",
        "reason": "Test inference did not complete in this run; test labels were not accessed.",
    }
    TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report, indent=2) + "\n")
else:
    TEST_LABELS = DATA_ROOT / "meta_data" / "test_average.csv"
    if not TEST_LABELS.is_file():
        test_metrics_report = {
            "status": "SKIPPED",
            "run_purpose": "research_only",
            "reason": "Official test labels are unavailable; no metrics were fabricated.",
        }
        TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report, indent=2) + "\n")
    else:
        from scripts.full7_test_metrics import evaluate_post_inference_metrics
        test_metrics_report = evaluate_post_inference_metrics(
            TEST_PREDICTION_SNAPSHOT,
            TEST_LABELS,
            TEST_METRICS_PATH,
            expected_count=EXPECTED_COUNTS["test"],
        )
        test_metrics_report["submission_file"] = str(SUBMISSION)
        test_metrics_report["submission_sha256"] = hashlib.sha256(SUBMISSION.read_bytes()).hexdigest()
        assert test_metrics_report["prediction_sha256"] == test_metrics_report["submission_sha256"]
        TEST_METRICS_PATH.write_text(json.dumps(test_metrics_report, indent=2, sort_keys=True) + "\n")
print("Post-inference test metrics:", test_metrics_report)


## 17 — Persist immutable outputs / optional **private-only** Kaggle backup

Always save a Kaggle Notebook Version with outputs. For cross-session recovery before Gate finishes, upload a **private** dataset only if your XACLE dataset/pretrained-resource agreements permit it. Backup defaults to local tar; `UPLOAD_PRIVATE_BACKUP=False` does not make external network calls. `Full9` is not repackaged because it already exists as a Kaggle Input.

In [ ]:
from scripts.kaggle_full9_artifacts import check_private_upload_target

BACKUP_ROOT = WORK_ROOT / "backup"
BACKUP_ROOT.mkdir(exist_ok=True)
ARCHIVE = BACKUP_ROOT / "full7_intermediates.tar"
if CREATE_LOCAL_BACKUP:
    with tarfile.open(ARCHIVE.with_suffix(".tar.tmp"), "w") as tar:
        for resource in (WHISPER_ROOT, E5_ROOT, E6_ROOT, FULL7_ROOT, GATE_ROOT, CONFIG_ROOT, LOG_ROOT):
            if not resource.exists():
                continue
            for f in resource.rglob("*"):
                if f.is_file():
                    tar.add(f, arcname=str(f.relative_to(WORK_ROOT)), recursive=False)
    ARCHIVE.with_suffix(".tar.tmp").replace(ARCHIVE)
    digest = hashlib.sha256()
    with ARCHIVE.open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""):
            digest.update(block)
    manifest = {"source_commit": SOURCE_COMMIT, "archive_sha256": digest.hexdigest(),
                "archive_size_bytes": ARCHIVE.stat().st_size, "full9_source": str(FULL9_ROOT),
                "contains_test_labels": False, "challenge_submission_approval_claimed": False}
    (BACKUP_ROOT / "backup_manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Local backup:", ARCHIVE, "GiB:", round(ARCHIVE.stat().st_size/2**30, 3))

if UPLOAD_PRIVATE_BACKUP:
    if not CREATE_LOCAL_BACKUP:
        raise ValueError("Enable CREATE_LOCAL_BACKUP before uploading")
    import kagglehub
    check_private_upload_target(PRIVATE_BACKUP_HANDLE)
    kagglehub.dataset_upload(handle=PRIVATE_BACKUP_HANDLE,
                             local_dataset_dir=str(BACKUP_ROOT),
                             version_notes="Full7 verified intermediate cache and checkpoint backup")
    print("Private backup uploaded:", PRIVATE_BACKUP_HANDLE)
print("Done. Check each stage's log file:", LOG_ROOT)
